# Layer 2: active learning with BanglaBERT on SentNoB (labels revealed from the gold column)

Colab notebook, GPU runtime. Saved outputs are cleared; the curves it produced are in `results/sentnob_layer2/`.

Cell order: install/import, load SentNoB, clean and split (200 seed rows, rest is the pool, locked test set), query strategies, BanglaBERT fine-tuning helper, full-supervised reference, AL loops (margin, random, entropy until a target F1 or patience runs out), plots.


**Step 1 — Switch runtime, then install/import**

In [ ]:
!pip install -q transformers datasets accelerate

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from google.colab import files
import pickle

In [ ]:
BASE_URL = "https://huggingface.co/datasets/sustcsenlp/bn_sentiment_noisy_dataset/resolve/main"

train_raw = pd.read_csv(f"{BASE_URL}/Train.csv")
val_raw   = pd.read_csv(f"{BASE_URL}/Val.csv")
test_raw  = pd.read_csv(f"{BASE_URL}/Test.csv")

print(train_raw.shape, val_raw.shape, test_raw.shape)

In [ ]:
def clean(df):
    df = df.rename(columns={"Data": "sentence", "Label": "sentiment"}).copy()
    df = df.dropna(subset=["sentence", "sentiment"])
    df["sentence"] = df["sentence"].astype(str).str.strip()
    df = df[(df["sentence"].str.len() > 1) & (df["sentence"].str.len() < 100)]
    df = df.drop_duplicates(subset="sentence").reset_index(drop=True)
    return df[["sentence", "sentiment"]]

pool_source = clean(pd.concat([train_raw, val_raw], ignore_index=True))
test_df     = clean(test_raw)

pool_source = pool_source[~pool_source["sentence"].isin(set(test_df["sentence"]))].reset_index(drop=True)

print("Pool source:", len(pool_source))
print("Locked test:", len(test_df))

In [ ]:
def make_split(pool_source, seed_size=200, seed=17):
    frac = seed_size / len(pool_source)
    seed_train = pool_source.groupby("sentiment", group_keys=False).sample(frac=frac, random_state=seed)
    pool_df = pool_source.drop(seed_train.index).reset_index(drop=True)
    return seed_train.reset_index(drop=True), pool_df

seed_train, pool_df = make_split(pool_source, seed_size=200, seed=17)
print("Seed train:", len(seed_train), dict(seed_train["sentiment"].value_counts()))
print("Pool:", len(pool_df))

In [ ]:
def least_confidence(probs):
    p_max = np.max(probs)
    n = len(probs)
    return -(1 - p_max) * (n / (n - 1))

def margin_sampling(probs):
    top2 = np.sort(probs)[::-1][:2]
    return top2[0] - top2[1]

def ratio_sampling(probs):
    top2 = np.sort(probs)[::-1][:2]
    return top2[0] / (top2[1] + 1e-12)

def entropy_sampling(probs):
    p = np.clip(probs, 1e-12, 1)
    ent = -np.sum(p * np.log2(p)) / np.log2(len(probs))
    return -ent

def random_sampling(probs, rng):
    return rng.random()

STRATEGIES = {
    "least_confidence": least_confidence,
    "margin": margin_sampling,
    "ratio": ratio_sampling,
    "entropy": entropy_sampling,
    "random": random_sampling,
}

**Step 2 — Load BanglaBERT's tokenizer once (reused every round)**

In [ ]:
MODEL_NAME = "csebuetnlp/banglabert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_texts(texts, max_length=64):
    return tokenizer(list(texts), truncation=True, padding=True, max_length=max_length, return_tensors="pt")

**Step 3 — A function that builds a fresh model each round**

In [ ]:
# def make_bert_model():
#     model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
#     return model.to(device)

from transformers import set_seed

def make_bert_model(seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    return model.to(device)

**Step 4 — One round's fine-tuning, wrapped as a function**

In [ ]:
def fine_tune_round(train_df, seed, epochs=3, batch_size=16):
    model = make_bert_model(seed)

    train_encodings = tokenize_texts(train_df["sentence"])
    train_dataset = Dataset.from_dict({
        "input_ids": train_encodings["input_ids"],
        "attention_mask": train_encodings["attention_mask"],
        "labels": train_df["sentiment"].tolist(),
    })

    # args = TrainingArguments(
    #     output_dir="/content/bert_tmp",
    #     num_train_epochs=epochs,
    #     per_device_train_batch_size=batch_size,
    #     logging_steps=50,
    #     save_strategy="no",
    #     report_to="none",
    #     seed=seed,
    # )

    args = TrainingArguments(
        output_dir="/content/bert_tmp",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=2e-5,      # down from the Trainer default of 5e-5
        warmup_ratio=0.1,        # ease into training instead of a hard start
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        seed=seed,
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
    trainer.train()
    return model

**Step 5 — Predict probabilities (for test scoring and pool querying)**

In [ ]:
import torch.nn.functional as F

def predict_probs(model, sentences, batch_size=32):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i+batch_size]
            enc = tokenize_texts(batch)
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)

In [ ]:
full_sup_model = fine_tune_round(pool_source, seed=17, epochs=3, batch_size=16)

test_probs_full = predict_probs(full_sup_model, test_df["sentence"].tolist())
test_preds_full = np.argmax(test_probs_full, axis=1)

bert_full_f1 = f1_score(test_df["sentiment"].values, test_preds_full, average="macro")
bert_full_acc = accuracy_score(test_df["sentiment"].values, test_preds_full)

print(f"BanglaBERT full-supervised: macro-F1={bert_full_f1:.3f}, accuracy={bert_full_acc:.3f}")

import pickle
with open("/content/bert_full_sup_result.pkl", "wb") as f:
    pickle.dump({"f1": bert_full_f1, "acc": bert_full_acc}, f)
files.download("/content/bert_full_sup_result.pkl")

**Step 6b — the AL loop function**

In [ ]:
def run_bert_active_learning(seed_train, pool_df, test_df, strategy_name,
                               n_rounds=15, batch_size=100, seed=17,
                               run_id=None):
    strategy_fn = STRATEGIES[strategy_name]
    rng = np.random.default_rng(seed)

    state_path = f"/content/bert_state_{run_id}.pkl"
    train_df, pool, logs, start_round = None, None, None, 0

    if os.path.exists(state_path):
        try:
            with open(state_path, "rb") as f:
                state = pickle.load(f)
            train_df = state["train_df"]
            pool = state["pool"]
            logs = state["logs"]
            start_round = state["round"] + 1
            print(f"Resuming {run_id} from round {start_round}")
        except (EOFError, pickle.UnpicklingError, KeyError) as e:
            print(f"Checkpoint file was corrupt/incomplete ({e}) — starting fresh instead.")

    if train_df is None:
        train_df = seed_train.copy().reset_index(drop=True)
        pool = pool_df.copy().reset_index(drop=True)
        logs = []
        start_round = 0

    for round_i in range(start_round, n_rounds):
        if len(pool) < batch_size:
            break

        model = fine_tune_round(train_df, seed=seed, epochs=3, batch_size=16)

        test_probs = predict_probs(model, test_df["sentence"].tolist())
        test_preds = np.argmax(test_probs, axis=1)
        test_f1_macro = f1_score(test_df["sentiment"].values, test_preds, average="macro")
        test_acc = accuracy_score(test_df["sentiment"].values, test_preds)

        logs.append({
            "strategy": strategy_name, "seed": seed, "round": round_i,
            "train_size": len(train_df),
            "test_accuracy": test_acc, "test_f1_macro": test_f1_macro,
        })
        print(f"[{run_id}] round {round_i} | train_size={len(train_df)} | test_f1_macro={test_f1_macro:.3f}")

        pool_probs = predict_probs(model, pool["sentence"].tolist())
        if strategy_name == "random":
            scores = np.array([strategy_fn(p, rng) for p in pool_probs])
        else:
            scores = np.array([strategy_fn(p) for p in pool_probs])

        order = np.argsort(scores)
        query_idx = order[:batch_size]
        queried = pool.iloc[query_idx]
        pool = pool.drop(pool.index[query_idx]).reset_index(drop=True)
        train_df = pd.concat([train_df, queried], ignore_index=True)

        with open(state_path, "wb") as f:
            pickle.dump({"round": round_i, "train_df": train_df, "pool": pool, "logs": logs}, f)


        del model
        torch.cuda.empty_cache()

    return pd.DataFrame(logs)

**Run AL with Margin sampling**

In [ ]:
run_id = "bert_margin_seed17"
result_margin = run_bert_active_learning(
    seed_train, pool_df, test_df,
    strategy_name="margin", n_rounds=25, batch_size=100, seed=17,
    run_id=run_id,
)
files.download(f"/content/bert_state_{run_id}.pkl")

Run AL with Random sampling

In [ ]:
run_id = "bert_random_seed17"
result_random = run_bert_active_learning(
    seed_train, pool_df, test_df,
    strategy_name="random", n_rounds=25, batch_size=100, seed=17,
    run_id=run_id,
)

**AL model function v2**

In [ ]:
def run_bert_active_learning_until_crossed(seed_train, pool_df, test_df, strategy_name,
                                             target_f1, batch_size=100, seed=17,
                                             run_id=None, max_rounds=60, patience=8):
    strategy_fn = STRATEGIES[strategy_name]
    rng = np.random.default_rng(seed)

    state_path = f"/content/bert_state_{run_id}.pkl"
    train_df, pool, logs, start_round = None, None, None, 0

    if os.path.exists(state_path):
        try:
            with open(state_path, "rb") as f:
                state = pickle.load(f)
            train_df = state["train_df"]
            pool = state["pool"]
            logs = state["logs"]
            start_round = state["round"] + 1
            print(f"Resuming {run_id} from round {start_round}")
        except (EOFError, pickle.UnpicklingError, KeyError) as e:
            print(f"Checkpoint file was corrupt/incomplete ({e}) — starting fresh instead.")

    if train_df is None:
        train_df = seed_train.copy().reset_index(drop=True)
        pool = pool_df.copy().reset_index(drop=True)
        logs = []
        start_round = 0

    best_f1_so_far = max([l["test_f1_macro"] for l in logs], default=0)
    rounds_since_improvement = 0

    round_i = start_round
    while round_i < max_rounds:
        if len(pool) < batch_size:
            print("Pool exhausted — stopping.")
            break

        model = fine_tune_round(train_df, seed=seed, epochs=3, batch_size=16)

        test_probs = predict_probs(model, test_df["sentence"].tolist())
        test_preds = np.argmax(test_probs, axis=1)
        test_f1_macro = f1_score(test_df["sentiment"].values, test_preds, average="macro")
        test_acc = accuracy_score(test_df["sentiment"].values, test_preds)

        logs.append({
            "strategy": strategy_name, "seed": seed, "round": round_i,
            "train_size": len(train_df),
            "test_accuracy": test_acc, "test_f1_macro": test_f1_macro,
        })
        print(f"[{run_id}] round {round_i} | train_size={len(train_df)} | test_f1_macro={test_f1_macro:.3f} | target={target_f1}")

        # --- crossing check ---
        if test_f1_macro >= target_f1:
            print(f"Crossed target {target_f1} at round {round_i}, train_size={len(train_df)}.")
            with open(state_path, "wb") as f:
                pickle.dump({"round": round_i, "train_df": train_df, "pool": pool, "logs": logs}, f)
            files.download(state_path)
            del model
            torch.cuda.empty_cache()
            break

        # --- plateau check ---
        if test_f1_macro > best_f1_so_far + 0.002:  # small tolerance for noise
            best_f1_so_far = test_f1_macro
            rounds_since_improvement = 0
        else:
            rounds_since_improvement += 1

        if rounds_since_improvement >= patience:
            print(f"No improvement in {patience} rounds (best={best_f1_so_far:.3f}) — stopping early, target not reached.")
            with open(state_path, "wb") as f:
                pickle.dump({"round": round_i, "train_df": train_df, "pool": pool, "logs": logs}, f)
            files.download(state_path)
            del model
            torch.cuda.empty_cache()
            break

        # --- query next batch, same as before ---
        pool_probs = predict_probs(model, pool["sentence"].tolist())
        if strategy_name == "random":
            scores = np.array([strategy_fn(p, rng) for p in pool_probs])
        else:
            scores = np.array([strategy_fn(p) for p in pool_probs])

        order = np.argsort(scores)
        query_idx = order[:batch_size]
        queried = pool.iloc[query_idx]
        pool = pool.drop(pool.index[query_idx]).reset_index(drop=True)
        train_df = pd.concat([train_df, queried], ignore_index=True)

        with open(state_path, "wb") as f:
            pickle.dump({"round": round_i, "train_df": train_df, "pool": pool, "logs": logs}, f)
        files.download(state_path)

        del model
        torch.cuda.empty_cache()
        round_i += 1

    return pd.DataFrame(logs)

**Entropy**

In [ ]:
run_id = "bert_entropy_seed17"
result_entropy = run_bert_active_learning_until_crossed(
    seed_train, pool_df, test_df,
    strategy_name="entropy", target_f1=0.72,
    batch_size=100, seed=17, run_id=run_id,
    max_rounds=60, patience=8,
)

**Pickle load**

In [ ]:
from google.colab import files
uploaded = files.upload()
# Select: bert_state_bert_margin_seed17.pkl, bert_state_bert_entropy_seed17.pkl,
#         bert_full_sup_result.pkl, and bert_state_bert_random_seed17.pkl if you have it

**Load them into DataFrames**

In [ ]:
import pickle
import pandas as pd

def load_al_logs(path):
    with open(path, "rb") as f:
        state = pickle.load(f)
    return pd.DataFrame(state["logs"])

results = {}

try:
    results["margin"] = load_al_logs("bert_state_bert_margin_seed17.pkl")
except FileNotFoundError:
    print("margin file not found — skipping")

try:
    results["entropy"] = load_al_logs("bert_state_bert_entropy_seed17.pkl")
except FileNotFoundError:
    print("entropy file not found — skipping")

try:
    results["random"] = load_al_logs("bert_state_bert_random_seed17.pkl")
except FileNotFoundError:
    print("random file not found — skipping (that's fine, upload later if you get it)")

with open("bert_full_sup_result.pkl", "rb") as f:
    full_sup = pickle.load(f)

print("Loaded strategies:", list(results.keys()))
print("Full-supervised:", full_sup)

**PLOT**

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

for strategy_name, df in results.items():
    ax.plot(df["train_size"], df["test_f1_macro"], marker="o", markersize=3, label=strategy_name)

ax.axhline(full_sup["f1"], color="black", linestyle="--", linewidth=1.5,
           label=f"Full supervised ({full_sup['f1']:.3f})")

ax.set_xlabel("Labeled training rows")
ax.set_ylabel("Test macro-F1")
ax.set_title("BanglaBERT active learning on SentNoB")
ax.legend()
plt.tight_layout()
plt.savefig("bert_al_comparison.png", dpi=150)
plt.show()

**Labels needed to reach 90% / 95% of ceiling (the table to report)**

In [ ]:
def labels_to_reach(df, target_f1):
    hit = df[df["test_f1_macro"] >= target_f1]
    return hit["train_size"].iloc[0] if len(hit) else None

rows = []
for strategy_name, df in results.items():
    for frac in [0.90, 0.95]:
        target = frac * full_sup["f1"]
        n = labels_to_reach(df, target)
        rows.append({
            "strategy": strategy_name,
            "target_pct_of_ceiling": f"{int(frac*100)}%",
            "target_f1": round(target, 3),
            "labels_needed": n if n else "not reached",
        })

pd.DataFrame(rows)